# Katydid Genus Spectrogram Viewer

A quick spot check rather than an exhaustive browse: pick a genus and see one
random spectrogram from it, with a button to roll a new one. Since
`katydid_final_multi` can have several rows per clip (one per burst type), all
of them get printed together along with the clip's overall `Burst_Pattern`.

For paging through every spectrogram instead, use `Display_Function.ipynb`.

**Run `Processing_Katydid_Spectrograms_Multiple_Bursts.ipynb` first.**

In [ ]:
import random
import re
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from PIL import Image

DISCRETE_SIGNALS_DIR = Path.home() / "Discrete_Signals"

In [ ]:
%store -r katydid_final_multi

In [ ]:
class KatydidGenusViewer:
    """Show one randomly chosen species + spectrogram from the current genus.
    Since katydid_final_multi has one row per (file, burst type), the chosen
    spectrogram may correspond to several rows; all are shown together.

    Matches images to rows via Spec_ID, not File_ID: File_ID is the audio
    source's own id (for traceability in the CSV) and generally won't appear
    in the local generated PNG filename, which is derived from the SINA
    spectrogram URL.
    """

    TAXON_DIR = "Katydids"
    IMAGE_GLOB = "{genus}_{species}_generated_spectrogram*"
    ID_PATTERN = r"_spectrogram_(.+)\.[^.]+$"

    def __init__(self, df):
        self.df = df.reset_index(drop=True)

        # Preserve first-appearance order of genera, and of species within each genus
        self.genus_list = []
        self.species_by_genus = {}
        for genus, species in zip(self.df["Genus"], self.df["Species"]):
            if genus not in self.species_by_genus:
                self.genus_list.append(genus)
                self.species_by_genus[genus] = []
            if species not in self.species_by_genus[genus]:
                self.species_by_genus[genus].append(species)

        self.genus_index = 0
        self.current_species = None
        self.current_image_path = None

        self.output = widgets.Output()
        self.prev_genus = widgets.Button(description="← Previous genus", layout=widgets.Layout(width="180px"))
        self.next_genus = widgets.Button(description="Next genus →", layout=widgets.Layout(width="180px"))
        self.randomize = widgets.Button(description="New random spectrogram", layout=widgets.Layout(width="200px"))
        self.prev_genus.on_click(lambda b: self.change_genus(-1))
        self.next_genus.on_click(lambda b: self.change_genus(1))
        self.randomize.on_click(lambda b: self.pick_random_and_show())

        display(widgets.HBox([self.prev_genus, self.next_genus]))
        display(self.randomize)
        display(self.output)
        self.pick_random_and_show()

    def image_dir(self, genus, species):
        return DISCRETE_SIGNALS_DIR / self.TAXON_DIR / f"{genus}_{species}"

    def find_images(self, genus, species):
        pattern = self.IMAGE_GLOB.format(genus=genus, species=species)
        return sorted(self.image_dir(genus, species).glob(pattern))

    def get_all_images_for_genus(self, genus):
        """List of (species, image_path) tuples for every spectrogram in this genus."""
        images = []
        for species in self.species_by_genus[genus]:
            for image_path in self.find_images(genus, species):
                images.append((species, image_path))
        return images

    def change_genus(self, direction):
        self.genus_index = min(max(self.genus_index + direction, 0), len(self.genus_list) - 1)
        self.pick_random_and_show()

    def pick_random_and_show(self):
        genus = self.genus_list[self.genus_index]
        images = self.get_all_images_for_genus(genus)
        if images:
            self.current_species, self.current_image_path = random.choice(images)
        else:
            self.current_species, self.current_image_path = None, None
        self.show()

    def get_rows_for_image(self, genus, species, image_path):
        """Every row matching image_path's Spec_ID (one per burst type detected
        in that clip); falls back to the species' first row."""
        species_rows = self.df[(self.df["Genus"] == genus) & (self.df["Species"] == species)]
        if species_rows.empty:
            return species_rows
        match = re.search(self.ID_PATTERN, image_path.name)
        spec_id = match.group(1) if match else None
        hit = species_rows[species_rows["Spec_ID"] == spec_id]
        if not hit.empty:
            return hit
        return species_rows.iloc[[0]]

    def show(self):
        with self.output:
            clear_output(wait=True)
            genus = self.genus_list[self.genus_index]
            n_genera = len(self.genus_list)

            if self.current_image_path is None:
                print(f"Genus {self.genus_index + 1}/{n_genera}: {genus}")
                print(f"\nNo spectrogram found for any species in this genus.")
                return

            species = self.current_species
            rows = self.get_rows_for_image(genus, species, self.current_image_path)
            if rows.empty:
                print(f"No data found for {genus} {species}.")
                return

            print(f"Genus {self.genus_index + 1}/{n_genera}: {genus}")
            print(f"Randomly chosen: {genus} {species}")
            if "Burst_Pattern" in rows.columns:
                pattern = rows.iloc[0]["Burst_Pattern"]
                n_types = rows.iloc[0]["N_Burst_Types_Detected"]
                print(f"Burst pattern: {pattern} ({n_types} burst type(s) detected)")
            print()

            for _, row in rows.iterrows():
                label = f"Burst type {row['Burst_Type']}" if "Burst_Type" in rows.columns else "Stats"
                print(f"  {label}:")
                print(f"    Element length:         {row['Element_Length']}")
                print(f"    Inter-element interval: {row['Inter-Element_Interval']}")
                print(f"    Inter-burst interval:   {row['Inter-Burst_Interval']}")
                print(f"    Elements per burst:     {row['Elements_Per_Burst']} "
                      f"(min {row['Min_Elements_Per_Burst']}, max {row['Max_Elements_Per_Burst']})")
                if "N_Bursts_This_Type" in rows.columns:
                    print(f"    N bursts this type:     {row['N_Bursts_This_Type']}")
                print()

            print(f"  Temperature: {rows.iloc[0]['Temperature']}")
            print(f"  Location:    {rows.iloc[0]['Location']}")

            try:
                image = Image.open(self.current_image_path)
                try:
                    image.seek(0)   # animated GIFs: show the first frame only
                except EOFError:
                    pass
            except Exception as e:
                print(f"Could not load image: {e}")
                return

            figure, axes = plt.subplots(figsize=(12, 4))
            axes.imshow(image, cmap="gray")
            axes.axis("off")
            plt.tight_layout()
            plt.show()
            plt.close(figure)

In [ ]:
katydid_genus_viewer = KatydidGenusViewer(katydid_final_multi)